### Mudanca na selecao de datas para entrar na rodada do Hants

incluido date range para selecao de datas

In [ ]:

# -*- coding: utf-8 -*-
import os
import glob
import re
import tarfile
import numpy as np
import gzip
import shutil
import multiprocessing
import time
import zipfile
import datetime
# import datetime as dt

import matplotlib.pyplot as plt
# import hants
np.set_printoptions(suppress=True)
import warnings

from osgeo import gdal, ogr, osr


reT = re.compile(r'.*?_sl_H.*?')

from sklearn.preprocessing import MinMaxScaler

import pandas as pd
import shapely
import geopandas as gpd
import pyproj

import datetime as dt


from rasterstats import zonal_stats

warnings.filterwarnings('ignore')

import sys
sys.path.insert(0,'/home/ubuntu/gaivota_codes/')

from gaivota_drivers.S3Management import S3Management 
from gaivota_drivers.SatelliteDriver import SatelliteDriver
from gaivota_drivers.SystemDriver import SystemDriver

sd=SystemDriver()
sat=SatelliteDriver()
s3=S3Management()

from pylab import rcParams
rcParams['figure.figsize'] =10,10

numProcess=int(multiprocessing.cpu_count() * .8)#-1)
numProcess=int(multiprocessing.cpu_count() * .25)#-1)


print(numProcess)



# MODIS Data - Pre-processing

## Resample MOD09Q1 bands - #rows,cols

In [ ]:
path_in='/media/data/projeto_monitoramento_v2/01_mod09q1/'
path_out='/media/data/projeto_monitoramento_v2/01_mod09q1_resampled/'
sd.check_dir(path_in)
sd.check_dir(path_out)

# Arquivo com ncols,nrows de referencia para manter padrao dos anos passados
reference='/media/data/projeto_monitoramento_v2/feature_evi2_reference.tif'
print('epsg',sat.get_raster_projection(reference))
print('resolution',sat.get_raster_resolution(reference))


numProcess=int(multiprocessing.cpu_count()-1)

years=sorted(next(os.walk(path_in))[1])
print(years)


# for year in years:
#     print '======',year,'======'
#     flist=sorted(glob.glob(os.path.join(path_in,year,'*.tif')))

#     path_out_f=  os.path.join(path_out,year)
#     sd.check_dir(path_out_f)  
#     print(year,len(flist),'files')
    
#     for f in range(len(flist)):
# #         print os.path.basename(flist[f])
            
#         outpath=os.path.join(path_out_f,os.path.basename(flist[f]))
# #                 outpath=os.path.join(path_out,outname)
#         print os.path.isfile(outpath),outpath
    
    
#         if(os.path.isfile(outpath)==False):

#                 ###sat.gdal_resample(flist[f],reference,outpath,type_res='bilinear')

#             multiprocessing.Process(target=sat.gdal_resample, 
#                          args=(flist[f],reference,outpath)).start()


#         while (len(multiprocessing.active_children()) > numProcess):
#             time.sleep(1)



# while (len(multiprocessing.active_children()) > 0):
#     time.sleep(1)

# print 'FINISHED!'




In [ ]:
# CALC NDVI FROM MOD09 BANDS 

def calculate_ndvi(nir_layer,red_layer,path_out,nodata=0.,scale=1.0,type_out='Float32'):
    print ('Calculating NDVI...')
    if(os.path.isfile(path_out)==False):
        sd.call_command('gdal_calc.py -A {} -B {} --outfile={} --calc="((0.0001*A-0.0001*B)/(0.0001*A+0.0001*B))*{}" --NoDataValue={} --type={}'.format(nir_layer,red_layer,path_out,scale,nodata,type_out))

def calculate_evi(nir_layer,red_layer,path_out,nodata=0.0,scale=1.0,type_out='Float32'):
    print ('Calculating EVI...')
    if(os.path.isfile(path_out)==False):
        sd.call_command('gdal_calc.py --co="COMPRESS=LZW" -A {} -B {} --outfile={} --calc="(2.5*(0.0001*A-0.0001*B)/(0.0001*A+(2.4*0.0001*B)+1))*{}" --NoDataValue={} --type={}'.format(nir_layer,red_layer,path_out,scale,nodata,type_out))

    
def doy_to_yyymmdd(date):
    
    dt = datetime.datetime(int(date[0:4]),1,1)
    dtdelta = datetime.timedelta(days=int(date[4:])-1)
    newdate=str((dt + dtdelta)).split(' ')[0]
    return newdate



features=[
#     'evi',
    'ndvi'
]

path='/media/data/projeto_monitoramento_v2/'
path_mod09= os.path.join(path,'01_mod09q1')



    
# numProcess=int(multiprocessing.cpu_count() * .8)#-1)

years=[
#     '2001'
#         '2014',
#         '2015',
#         '2016',
#         '2017',
#         '2018',
# '2019',
# '2020',
# '2021',
# '2022',
'2023',
    '2024',
]
    
    
    


for year in years:
    print '======',year,'======'
    b1_list=sorted(glob.glob(os.path.join(path_mod09,str(year),'MOD09Q1*b01*.tif')))
    
    b2_list=sorted(glob.glob(os.path.join(path_mod09,str(year),'MOD09Q1*b02*.tif')))
    print ('b1_list',len(b1_list))
    print ('b1_list',len(b2_list))

    for feature in features:


        path_out=  os.path.join(path,'02_mod09q1_'+feature,year)
        sd.check_dir(path_out)   
    
        for f in range(len(b1_list)):
            print (os.path.basename(b1_list[f]))
            print (os.path.basename(b2_list[f]))

            date=os.path.basename(b1_list[f]).split('_')[4].replace('doy','')
            date2=os.path.basename(b2_list[f]).split('_')[4].replace('doy','')
            print (date,date==date2)
            if(date==date2):
                newdate=doy_to_yyymmdd(date)
                print (newdate)
#                 outname=os.path.basename(b1_list[f]).replace('.061_sur_refl_b01_doy','_'+feature+'_').replace('_aid0001','')
                outname=os.path.basename(b1_list[f]).split('.')[0]+'_'+feature+'_'+newdate+'.tif'
    
                outname=outname.replace(date,newdate)
                print outname
                outpath=os.path.join(path_out,outname)
                print (os.path.isfile(outpath),outpath)
                print ('============================')
        
                if(os.path.isfile(outpath)==False):
                    if(feature=='ndvi'):
                        multiprocessing.Process(target=calculate_ndvi, 
                                     args=(b2_list[f],b1_list[f],outpath,0.,10000,'Int16')).start()

                    if(feature=='evi'):
                        multiprocessing.Process(target=calculate_evi, 
                                     args=(b2_list[f],b1_list[f],outpath,0.,10000,'Int16')).start()


                while (len(multiprocessing.active_children()) > numProcess):
                    time.sleep(1)



while (len(multiprocessing.active_children()) > 0):
    time.sleep(1)
# MOD09Q1_ndvi_2022-09-14.tif
print('FINISHED!')

In [ ]:
for feature in features:
    for year in years:
        print ('======',year,'======')
        flist=sorted(glob.glob(os.path.join(path,'02_mod09q1_'+feature,year,'*.tif')))
        print(len(flist))
        
        for f in flist:
            try:
                res=sat.get_raster_resolution(f)
                print(feature,year,os.path.basename(f),res)
            except:
                print('ERROR',feature,year,os.path.basename(f),res)
                
                
                
# ('ndvi', '2023', 'MOD09Q1_ndvi_2022-08-29.tif', 0.0020833333331466974)


In [ ]:
# Resample FEATURE to BR old shape


reference='/media/data/projeto_monitoramento_v2/feature_evi2_reference.tif'




for feature in features:
    for year in years:
        print ('======',year,'======')
        flist=sorted(glob.glob(os.path.join(path,'02_mod09q1_'+feature,year,'*.tif')))

        path_out=  os.path.join(path,'02_mod09q1_'+feature+'_resampled/',year)
        sd.check_dir(path_out)   
    
        for f in range(len(flist)):
#             print os.path.basename(flist[f])
            
            outpath=os.path.join(path_out,os.path.basename(flist[f]))
#                 outpath=os.path.join(path_out,outname)
            print (os.path.isfile(outpath),outpath.split(path)[-1])
    
    
            if(os.path.isfile(outpath)==False):
            
                ###sat.gdal_resample(flist[f],reference,outpath,type_res='bilinear')

                multiprocessing.Process(target=sat.gdal_resample, 
                             args=(flist[f],reference,outpath)).start()


            while (len(multiprocessing.active_children()) > numProcess):
                time.sleep(1)



while (len(multiprocessing.active_children()) > 0):
    time.sleep(1)

print ('FINISHED!')

In [ ]:
# states=[
# #     'BA',
# #     'CE',
# #     'SP','MG','ES','RJ',
# #     'GO','DF','MT','MS',
# #     'PR','SC','RS','PRY',
# # 'MA','PI','RN','PB','AL','SE','PE',
# # 'PA', 'RO','RR','AP','AM','AC','TO',
    
#     'PRY',
# #     'MT',
# #     'GO','MS','DF',
# ]

# states=sorted(states)
# print(len(np.unique(states)))

In [ ]:

def gaivota_clip_raster(tif,shp,outname,nodata=0.):
        ds=gdal.Open(tif)
        gt=ds.GetGeoTransform()
        df=gpd.read_file(shp)
        bounds=df.total_bounds
#         print bounds

        # Get shapefile bounds in array col,row units and adjust to temp clip
        minx= (float(bounds[0])-gt[0])/gt[1]
        minx_col=int(minx)-1

        maxx=(float(bounds[2])-gt[0])/gt[1]
        maxx_col=int(maxx)+1

        maxy=(float(bounds[3])-gt[3])/gt[5]
        maxy_row=int(maxy)-1

        miny=(float(bounds[1])-gt[3])/gt[5]
        miny_row=int(miny)+1

        new_minx=gt[0]+minx_col*gt[1]
        new_maxx=gt[0]+maxx_col*gt[1]
        new_miny=gt[3]+miny_row*gt[5]
        new_maxy=gt[3]+maxy_row*gt[5]

        # Create temp raster clipped by adjusted bounds
        temp_tif=outname.replace('.tif','_temp.tif')
        os.system('gdal_translate -projwin {} {} {} {} -co "COMPRESS=LZW" -of GTiff {} {}'.format(new_minx,new_maxy,new_maxx,new_miny,os.path.abspath(tif),temp_tif))

        # get raster resolution
        res=sat.get_raster_resolution(temp_tif)

        # Clip temp raster to shapefile polygon - DO NOT USE -crop_to_cutline (prevents pixel grid shift)
        os.system('gdalwarp -dstnodata {} -q -cutline {} -tr {} {} -co "COMPRESS=LZW" -of GTiff {} {}'.format(nodata,shp,res,res,os.path.abspath(temp_tif),outname))
        os.remove(temp_tif)
        return outname


numProcess = int(multiprocessing.cpu_count())



path='/media/data/projeto_monitoramento_v2/'

states=[
    'BA',
    'CE',
    'SP','MG','ES','RJ',
    'GO',
    'DF',
    'MT',
    'MS',
    'PR','SC','RS','PRY',
'MA','PI','RN','PB','AL','SE','PE',
'PA', 'RO','RR','AP','AM','AC','TO',
  'PRY',  
    'URY',

#     'ARG',
    
    
]
states=np.unique(states)

features=[
#     'evi',
    'ndvi'
]

years=[
#     '2014',
#     '2015',
#     '2016',
#     '2017',
#     '2018',
#     '2019',
#     '2020',
#     '2021',
#     '2022',
    '2023',
    '2024',
]
  


for feature in features:
    
    for st in states:
        for year in years:
#             path_out=  os.path.join(path,'02_mod09q1_'+feature+'_resampled',year)


            ###sem resample
            path_out=  os.path.join(path,'02_mod09q1_'+feature,year)

            shp=glob.glob(os.path.join('/home/ubuntu/gaivota_shapes/BR_states_PY_URY/BR_'+st+'/','*.shp'))[0]
            print ('shp:',os.path.isfile(shp),shp)

            path_clip=  os.path.join(path,'03_mod09q1_'+feature+'_clip',st)
            sd.check_dir(path_clip)

            flist=sorted(glob.glob(os.path.join(path_out,'*.tif')))

            print (len(flist))

            for i in range(len(flist)):
                outname=os.path.join(path_clip,os.path.basename(flist[i]).replace('.tif','_clip_'+st+'.tif'))
        #         print f
                print (str(i).zfill(2),'of',len(flist),'-',outname)#os.path.basename(outname)
#                 print '====================================='
                
                if(os.path.isfile(outname)==False):
            
                    ######gaivota_clip_raster(flist[i],shp,outname)
                    
                    multiprocessing.Process(target=gaivota_clip_raster, 
                                 args=(flist[i],shp,outname)).start()
                    while (len(multiprocessing.active_children()) > numProcess):
                        time.sleep(1)


    
while (len(multiprocessing.active_children()) > 0):
    
    time.sleep(1)
    
print ('FINISHED!')

In [ ]:
"""
LIMPAR DADOS INVALIDOS NO NDVI
"""

def call_clear_geotiff(f,outname_temp,outname):
    _,arr,gt,proj=sat.raster_2_array(f)
    arr=arr.astype(float)
    arr[arr<0]=0
    arr[arr>10000]=10000

    
    sat.array_2_raster(outname_temp,arr,gt,proj,nodata=0,type_data='int16')
    os.system('gdal_translate -a_nodata 0 -co "COMPRESS=LZW" -of GTiff '+outname_temp+' '+outname)
    os.remove(outname_temp)

    
    
# features=['evi']

numProcess = int(multiprocessing.cpu_count() / 4.)
numProcess=1
    
for feature in features:
    
    # definir diretorio de entrada e saida
    path_clip='/media/data/projeto_monitoramento_v2/03_mod09q1_'+feature+'_clip/'
    path_clear='/media/data/projeto_monitoramento_v2/04_mod09q1_'+feature+'_clear/'
    sd.check_dir(path_clear)
    
    
    for folder in sorted(next(os.walk(path_clip))[1]):
        flist=sorted(glob.glob(os.path.join(path_clip,folder,'*.tif')))
        print folder, len(flist),'files'


        for f in flist:

            path_out_folder=os.path.join(path_clear,folder)
            sd.check_dir(path_out_folder)

            outname_temp=os.path.join(path_out_folder,os.path.basename(f).replace('.tif','_temp.tif'))
            outname=os.path.join(path_out_folder,os.path.basename(f))
            print( f)
    #         print outname_temp
            print (os.path.isfile(outname),outname)
            print ('---')
            if(os.path.isfile(outname)==False):
                ###call_clear_geotiff(f,outname_temp,outname)
                multiprocessing.Process(target=call_clear_geotiff, 
                                    args=(f,outname_temp,outname)).start()


                while (len(multiprocessing.active_children()) > numProcess):
                    time.sleep(1)


        print '==========================================================================='
while (len(multiprocessing.active_children()) > 0):
    
    time.sleep(1)    
    
print 'FINISHED!'

In [ ]:
"""
MOVER ARQUIVOS PARA A ESTRUTURA DE PASTAS PADRAO DO HANTS feature/state/year
"""

# path='/home/ubuntu/bruno_testes/2019-07-04_MT_Cotton_download_NDVI_MODIS_hants/'
path='/media/data/projeto_monitoramento_v2/'

flist=sorted(glob.glob(os.path.join(path,'*.tif')))

# features=['ndvi']

# states=['MS']

# states=['SP','GO','DF','MT','MS','PR','SC','RS','MG','BA','TO','MA','PI']

for state in states:
    for fea in features:

        flist=sorted(glob.glob(os.path.join(path,'04_mod09q1_'+fea+'_clear',state,'*.tif')))
        print state,len(flist),'files to move'

        for f in flist:
            print (os.path.isfile(f),f)
            split=os.path.basename(f).split('_')
            print (split)
            year=split[2][0:4]
            path_out=os.path.join(path,fea,state,year)
            print (os.path.isdir(path_out),path_out)
            sd.check_dir(path_out)
            newname=split[2].replace('-','_')+'_'+split[0]+'_'+fea+'_clip_BR_'+state+'.tif'
#             print newname
            newpath=os.path.join(path_out,newname)
            print (os.path.isfile(newpath),newpath)
        
            if(os.path.isfile(newpath)==False):
                shutil.copy(f,newpath)
            print'---'
        
print ('FINISHED!')

# Run Hants - START

## Hants Functions

In [ ]:
def create_virtual_raster(flist,outname, srcnodata=0):
    flist_str=''
    for i in range(len(flist)):
        flist_str=flist_str+' '+flist[i]
#         print os.path.basename(flist[i])
#     flist_str=flist_str[:-1]
        
    os.system('gdalbuildvrt -separate -srcnodata {} {} {}'.format(srcnodata,os.path.abspath(outname),flist_str))
    return outname

def gdal_array_2_tif(input_array,gt,proj,outname):
    nrows,ncols = input_array.shape
    output_raster = gdal.GetDriverByName('GTiff').Create(outname,ncols, nrows, 1 ,gdal.GDT_Float32, options = [ 'COMPRESS=LZW' ])
    output_raster.SetGeoTransform(gt)
    srs = osr.SpatialReference()
    srs.ImportFromEPSG(4326)
    output_raster.SetProjection( srs.ExportToWkt() )
    output_raster.GetRasterBand(1).WriteArray(input_array)
    return outname

def gdal_tif_2_array(raster):
    dataset = gdal.Open(raster,  gdal.GA_ReadOnly)
#     dataset_array = dataset.GetRasterBand(1).ReadAsArray()
    dataset_array=dataset.ReadAsArray()
    proj = dataset.GetProjection()
    gt = dataset.GetGeoTransform()  
    return dataset_array, gt, proj

def stack_tifs_as_arrays(fileslist):
    array,gt,proj =gdal_tif_2_array(fileslist[0])   
    for i in range(1,len(fileslist)):
        array=np.dstack((array, gdal_tif_2_array(fileslist[i])[0]))
    return array,gt,proj

# def get_sentinel_tilename(s2_tiles,shp):
#     tiles_df=gpd.read_file(s2_tiles)
#     shp_df=gpd.read_file(shp)
#     gettile=gpd.overlay(shp_df, tiles_df, how='intersection')
#     tiles=np.unique(gettile.NAME)
#     print tiles
#     return tiles

# def get_landsat_tilename(tiles,shp):
#     tiles_df=gpd.read_file(tiles)
#     shp_df=gpd.read_file(shp)
#     gettile=gpd.overlay(shp_df, tiles_df, how='intersection')
#     tiles=np.unique(gettile.PR)
#     print tiles
#     return tiles

def plot_array(array1,cmap1):
    plt.figure()
    imgplot = plt.imshow(array1)
    plt.set_cmap(cmap1)
    plt.colorbar(ticks=[np.nanmin(array1), np.nanmean(array1), np.nanmax(array1)], 
                 orientation='vertical')
    plt.show()
    fig=None
    
def plot_arrays(array1,array2,cmap1='jet',cmap2='jet'):
    fig = plt.figure()
    a = fig.add_subplot(1, 2, 1)
    imgplot = plt.imshow(array1)
    plt.set_cmap(cmap1)
    a.set_title('Array1')
    plt.colorbar(ticks=[np.nanmin(array1), np.nanmean(array1), np.nanmax(array1)], 
                 orientation='vertical')
    b = fig.add_subplot(1, 2, 2)
    imgplot = plt.imshow(array2)
    plt.set_cmap(cmap2)
    b.set_title('Array2')
    plt.colorbar(ticks=[np.nanmin(array2), np.nanmean(array2), np.nanmax(array2)], 
                 orientation='vertical')
    fig.subplots_adjust(left=0, bottom=0, right=1.8, top=1.8, wspace=None, hspace=None)
    plt.show()
    fig=None
    imgplot=None

#######################################################################################################
#######################################################################################################
#######################################################################################################

# HANTS

# Computing diagonal for each row of a 2d array. See: http://stackoverflow.com/q/27214027/2459096
def makediag3d(M):
    b = np.zeros((M.shape[0], M.shape[1] * M.shape[1]))
    b[:, ::M.shape[1] + 1] = M
    return b.reshape(M.shape[0], M.shape[1], M.shape[1])


def get_starter_matrix(base_period_len, sample_count, frequencies_considered_count):
    nr = min(2 * frequencies_considered_count + 1,
                  sample_count)  # number of 2*+1 frequencies, or number of input images
    mat = np.zeros(shape=(nr, sample_count))
    mat[0, :] = 1
    ang = 2 * np.pi * np.arange(base_period_len) / base_period_len
    cs = np.cos(ang)
    sn = np.sin(ang)
    # create some standard sinus and cosinus functions and put in matrix
    i = np.arange(1, frequencies_considered_count + 1)
    ts = np.arange(sample_count)
    for column in xrange(sample_count):
        index = np.mod(i * ts[column], base_period_len)
        # index looks like 000, 123, 246, etc, until it wraps around (for len(i)==3)
        mat[2 * i - 1, column] = cs.take(index)
        mat[2 * i, column] = sn.take(index)
    return mat


def HANTS(sample_count, inputs,frequencies_considered_count=3,
          outliers_to_reject='Hi',low=0., high=255,
          fit_error_tolerance=5,delta=0.1):
    """
    Function to apply the Harmonic analysis of time series applied to arrays
    sample_count    = nr. of images (total number of actual samples of the time series)
    base_period_len    = length of the base period, measured in virtual samples
            (days, dekads, months, etc.)
    frequencies_considered_count    = number of frequencies to be considered above the zero frequency
    inputs     = array of input sample values (e.g. NDVI values)
    ts    = array of size sample_count of time sample indicators
            (indicates virtual sample number relative to the base period);
            numbers in array ts maybe greater than base_period_len
            If no aux file is used (no time samples), we assume ts(i)= i,
            where i=1, ..., sample_count
    outliers_to_reject  = 2-character string indicating rejection of high or low outliers
            select from 'Hi', 'Lo' or 'None'
    low   = valid range minimum
    high  = valid range maximum (values outside the valid range are rejeced
            right away)
    fit_error_tolerance   = fit error tolerance (points deviating more than fit_error_tolerance from curve
            fit are rejected)
    dod   = degree of overdeterminedness (iteration stops if number of
            points reaches the minimum required for curve fitting, plus
            dod). This is a safety measure
    delta = small positive number (e.g. 0.1) to suppress high amplitudes
    """
    # define some parameters
    base_period_len = sample_count  #
    # check which setting to set for outlier filtering
    if outliers_to_reject == 'Hi':
        sHiLo = -1
    elif outliers_to_reject == 'Lo':
        sHiLo = 1
    else:
        sHiLo = 0
    nr = min(2 * frequencies_considered_count + 1,
             sample_count)  # number of 2*+1 frequencies, or number of input images
    # create empty arrays to fill
    outputs = np.zeros(shape=(inputs.shape[0], sample_count))
    mat = get_starter_matrix(base_period_len, sample_count, frequencies_considered_count)
    # repeat the mat array over the number of arrays in inputs
    # and create arrays with ones with shape inputs where high and low values are set to 0
    mat = np.tile(mat[None].T, (1, inputs.shape[0])).T
    p = np.ones_like(inputs)
    p[(low >= inputs) | (inputs > high)] = 0
    nout = np.sum(p == 0, axis=-1)  # count the outliers for each timeseries
    # prepare for while loop
    ready = np.zeros((inputs.shape[0]), dtype=bool)  # all timeseries set to false
    dod = 1  # (2*frequencies_considered_count-1)  # Um, no it isn't :/
    noutmax = sample_count - nr - dod
    for _ in xrange(sample_count):
        if ready.all():
            break
        # print '--------*-*-*-*',it.value, '*-*-*-*--------'
        # multiply outliers with timeseries
        za = np.einsum('ijk,ik->ij', mat, p * inputs)
        # multiply mat with the multiplication of multiply diagonal of p with transpose of mat
        diag = makediag3d(p)
        A = np.einsum('ajk,aki->aji', mat, np.einsum('aij,jka->ajk', diag, mat.T))
        # add delta to suppress high amplitudes but not for [0,0]
        A = A + np.tile(np.diag(np.ones(nr))[None].T, (1, inputs.shape[0])).T * delta
        A[:, 0, 0] = A[:, 0, 0] - delta
        # solve linear matrix equation and define reconstructed timeseries
        zr = np.linalg.solve(A, za)
        outputs = np.einsum('ijk,kj->ki', mat.T, zr)
        # calculate error and sort err by index
        err = p * (sHiLo * (outputs - inputs))
        rankVec = np.argsort(err, axis=1, )
        # select maximum error and compute new ready status
        maxerr = np.diag(err.take(rankVec[:, sample_count - 1], axis=-1))
        ready = (maxerr <= fit_error_tolerance) | (nout == noutmax)
        # if ready is still false
        if not ready.all():
            j = rankVec.take(sample_count - 1, axis=-1)
            p.T[j.T, np.indices(j.shape)] = p.T[j.T, np.indices(j.shape)] * ready.astype(int)#*check
            nout += 1
    return outputs,zr

def array_in(y):
    y = np.tile(y[None].T, (1, 1)).T
    kl=np.ones([1,1])
    y = y * kl
    return y


#######################################################################################################
#######################################################################################################
#######################################################################################################
    
def create_gaivota_tiles(totalbounds=[-180.,180.,-90.,90.],stepsize=10.):
    """
    
    Cria N tiles de 'stepsize' graus para recortar dados e rodar hants em paralelo
    
    totalbounds=[-xmin.,xmax.,-ymin.,ymax.] in degrees
    stepsize= in degrees
    EPSG: 4326 default
    """
    grid=gpd.GeoDataFrame(columns=['gtile_name','xmin','xmax','ymin','ymax','geometry'],geometry='geometry')
    grid.crs={'init': 'epsg:4326'}

    minx= totalbounds[0]+stepsize/2
    maxx= totalbounds[1]-stepsize/2
    miny= totalbounds[2]+stepsize/2
    maxy= totalbounds[3]-stepsize/2

    nw = shapely.geometry.Point((minx, maxy))
    se = shapely.geometry.Point((maxx, miny))

    gridpoints = []
    x = nw.x
    while x <= se.x:
        y = nw.y
        while y >= se.y:
    #         print x,y
            p = shapely.geometry.Point((x,y))
            gridpoints.append(p)
            y += -stepsize
        x += stepsize

    # create points - geodataframe
    print ('creating point grid...',)
    for i in range(len(gridpoints)):
        if(i%500==0):
            print(i,'of',len(gridpoints))
        grid.loc[i,'geometry']=gridpoints[i]
        
#         gname='gtile_'
        gname=''
        if(grid.loc[i,'geometry'].x<0):
            gname=gname+str((grid.loc[i,'geometry'].x)).replace('-','')+'W_'
        else:
            gname=gname+str((grid.loc[i,'geometry'].x))+'E_'
            
        if(grid.loc[i,'geometry'].y<0):
            gname=gname+str((grid.loc[i,'geometry'].y)).replace('-','')+'S'
        else:
            gname=gname+str((grid.loc[i,'geometry'].y))+'N'

        grid.loc[i,'gtile_name']=gname
    print ('Points created:',len(gridpoints)) 
    
    
    #points to polygons
    grid['geometry']=grid['geometry'].buffer(stepsize/2)
       
    print ('\ncreating polygon grid...',)
    for i in range(len(grid)):
        if(i%500==0):
            print(i,'of',len(grid))
        bounds=grid.loc[i,'geometry'].bounds
        grid.loc[i,'xmin']=bounds[0]
        grid.loc[i,'ymin']=bounds[1]
        grid.loc[i,'xmax']=bounds[2]
        grid.loc[i,'ymax']=bounds[3]

        p1 = shapely.geometry.Point(bounds[0],bounds[1])
        p2 = shapely.geometry.Point(bounds[0],bounds[3])
        p3 = shapely.geometry.Point(bounds[2],bounds[3])
        p4 = shapely.geometry.Point(bounds[2],bounds[1])
        pointList = [p1, p2, p3, p4]
        poly = shapely.geometry.Polygon([[p.x, p.y] for p in pointList])
        grid.loc[i,'geometry']=  poly
    print ('Polygons created:',len(grid))
    
    return grid

def select_gtiles_by_raster(grid_path,grid_df,raster):
    """
    use raster to select gaivota tiles
    """
#     grid_df=gpd.read_file(shp)
    statistics=['count']
    z_stats=zonal_stats(grid_path, raster,stats=statistics,nodata=0.)
    grid_df = grid_df.join(pd.DataFrame(z_stats))
    select_df=grid_df[grid_df['count']>0].reset_index(drop=True)
    return select_df

def select_gtiles_by_shp(shp_tiles,grid_df,shp_state):
    """
    shp_tiles - gaivota_tiles created with 'create_gaivota_tiles' (EPSG:4326)
    shp_st - state shapefile (must be in EPSG:4326)
    
    """
#     grid_df=gpd.read_file(shp_tiles)
    df_st=gpd.read_file(shp_state)
    # Spatial join
    df_select=gpd.sjoin(grid_df,df_st,how='inner')
    # delete shp_st columns
    df_select=df_select[grid_df.columns].reset_index(drop=True)
    return df_select
    
    

#######################################################################################################
#######################################################################################################
#######################################################################################################

# second part - appling hants for each states of regions
def call_hants_parallel(year,st,fea,gtile,flist,path_hants,feaToMinLimit,freq,
                        freq_recon,feaToErrorLimit,feaToOutlier, recon=True):
    initime=datetime.datetime.now()
    print('Start hants - state=',st,'year=',year,'gtile=',gtile)
    #==========================================================================
    #loading data

    array,gt,proj=sat.stack_tifs_as_arrays(flist)
#     for item in flist:
#         os.remove(item)
    
    array=array.astype(float)
    if 'ndmi'==fea or 'ndwi'==fea:
        scaler = MinMaxScaler(feature_range=(0, 10000.0))
        for k in range(array.shape[2]):
            array[:,:,k]=scaler.fit_transform(array[:,:,k])
    else:            
        array[array<feaToMinLimit[fea]]=feaToMinLimit[fea]        
#     print(array.max())
    array=array/10000.0   

    #==========================================================================
    # appling hants 
    zr_arr=np.zeros([array.shape[0],array.shape[1],1+2*freq],dtype=float)
#     result_arr=np.zeros([array.shape[0],array.shape[1],array.shape[2]],dtype=float)        
    print('array.shape=',array.shape)
    print('zr_arr.shape=',zr_arr.shape)

    for i in range(array.shape[0]):
        if i%100==0:
            print(' appling hants i = ',i,' of ',array.shape[0],'rows', 'year='+year,'gtile='+gtile)
        for j in range(array.shape[1]):
            serie = array[i,j,:]
            if serie.max()>0.0:
                y = array_in(serie)
                result,zr=HANTS(sample_count=y.shape[1], inputs=y, frequencies_considered_count=freq, 
                                outliers_to_reject=feaToOutlier[fea], low=feaToMinLimit[fea], high=1., fit_error_tolerance=feaToErrorLimit[fea])
                zr_arr[i,j]=zr[0]
#                 result_arr[i,j]=result
            else:
                zr_arr[i,j]=0.0
#                 result_arr[i,j]=0.0
    
    array_shape=array.shape
    array=None
    #==========================================================================
    # saving results - amp and phase         
    amp=np.zeros([array_shape[0],array_shape[1],freq+1],dtype=float)
    phase=np.zeros([array_shape[0],array_shape[1],freq+1],dtype=float)
    amp[:,:,0]=zr_arr[:,:,0]
    print('AMP.shape=',amp.shape)
    print('PHASE.shape=',phase.shape)        
    path_feature_hants=os.path.join(path_hants,'amp_pha_'+str(freq)+'freq')
    path_feature_hants_res=os.path.join(path_hants,'reconstructed_'+str(freq_recon)+'freq')
    sd.check_dir(path_feature_hants)
    sd.check_dir(path_feature_hants_res)

    # save amp0 and ph0
    outname=os.path.join(path_feature_hants,'hants_am0_'+fea+'_'+st+'_'+year+'_'+gtile+'.tif')
    sat.array_2_raster(outname,amp[:,:,0],gt,proj)
    outname=os.path.join(path_feature_hants,'hants_ph0_'+fea+'_'+st+'_'+year+'_'+gtile+'.tif')
    sat.array_2_raster(outname,phase[:,:,0],gt,proj)

    # save N frequencies
    for f in range(1,2*freq+1,2):
        i=((f-1)/2)+1 # freq number
        print('saving harmonic:',i,'year='+year,'gtile='+gtile)
        zr_real=zr_arr[:,:,f]
        zr_img=zr_arr[:,:,f+1]
        amp[:,:,i]=(zr_real**2+zr_img**2)**0.5
        for m in range(amp.shape[0]):
            for n in range(amp.shape[1]):
                if(zr_real[m,n]==0):
                    phase[m,n,i]=(np.pi/2)*180.0/np.pi
                elif(zr_real[m,n]<0):
                    phase[m,n,i]=(np.arctan(zr_img[m,n]/zr_real[m,n])+np.pi)*(180.0/np.pi)
                else:
                    phase[m,n,i]=(np.arctan(zr_img[m,n]/zr_real[m,n]))*(180.0/np.pi)
                if(phase[m,n,i]<0):
                    phase[m,n,i]=phase[m,n,i]+360.0
        outname=os.path.join(path_feature_hants,'hants_am'+str(i)+'_'+fea+'_'+st+'_'+year+'_'+gtile+'.tif')
        #gdal_array_2_tif(amp[:,:,i],gt,proj,outname)
        sat.array_2_raster(outname,amp[:,:,i],gt,proj)
        outname=os.path.join(path_feature_hants,'hants_ph'+str(i)+'_'+fea+'_'+st+'_'+year+'_'+gtile+'.tif')
        #gdal_array_2_tif(phase[:,:,i],gt,proj,outname)
        sat.array_2_raster(outname,phase[:,:,i],gt,proj)
        
    zr_arr=None
    
    if(recon==True):
        #==========================================================================
        # recompose time series
        yrecon=np.zeros(array_shape)

        # AMP and PHASE resized to freq_recon+1 (axis 2)
        amp=amp[:,:,0:freq_recon+1]
        phase=phase[:,:,0:freq_recon+1]

    #     print 'amp_recon.shape:',amp.shape
    #     print 'phase_recon.shape:',phase.shape

        print('yrecon.shape=',yrecon.shape)
        for i in range(yrecon.shape[0]):
            if i%100==0:
                print('recompose time series: i = ',i,' of ',yrecon.shape[0],'rows',gtile)
            for j in range(yrecon.shape[1]):
                yrecon[i,j,:]=(np.tile(amp[i,j,:],(yrecon.shape[2], 1)).T*np.cos((np.tile(np.arange(yrecon.shape[2]),(freq_recon+1, 1))*np.arange(freq_recon+1).reshape((-1,1))*2.0*np.pi/yrecon.shape[2])-np.tile(phase[i,j,:]*np.pi/180.0,(yrecon.shape[2],1)).T)).sum(axis=0)

        amp=None
        phase=None
        #==========================================================================
        # saving recomposition of time series
        yrecon[yrecon>1]=1
        yrecon[yrecon<feaToMinLimit[fea]]=feaToMinLimit[fea]

        vrt_list=[]
        print ('reconstructing',yrecon.shape[2],'files', 'year='+year,gtile)
        for z in range(yrecon.shape[2]):

            fname=os.path.basename(flist[z])
            split=fname.split('_')            
            date_str=split[0]+'_'+split[1]+'_'+split[2]
            #"{0:0=3d}".format(z+1)
            #gdal_array_2_tif(yrecon[:,:,z]*10000,gt,proj,os.path.join(path_feature_hants_res,'yrecon'+str(freq_recon)+'_'+band+'.tif'))
            outname=os.path.join(path_feature_hants_res,os.path.basename(flist[z].replace('.tif','_hantsrecon_'+str(freq_recon)+'freq.tif')))
            vrt_list.append(outname)
            sat.array_2_raster(outname,yrecon[:,:,z]*10000,gt,proj,nodata=-28672,type_data='int16')

        yrecon=None

        print('vrt_list:',len(vrt_list),'files')
        vrt_recon=os.path.join(path_hants,fea+'_'+st+'_'+year+'_hants_'+str(freq_recon)+'freq_'+gtile+'.vrt')
        vrt=create_virtual_raster(vrt_list,vrt_recon)
    
    
    endtime=datetime.datetime.now()
    print '=== Hants finished! - state=',st,'year=',year,'gtile=',gtile, 'elapsed time=',endtime-initime
#     print '=== ELAPSED TIME:',endtime-initime,'gtile:'+gtile,'year='+year,'==='
        

###########################################################
def call_merge_gtiles(flist,shp_st,outname,nodata=0.,remove_files=False):
    tempname=outname.replace('.tif','_merge.tif')
    str_list=''
    for item in flist:
        str_list=str_list+' '+item
    str_list=str_list[1:]

    os.system('gdal_merge.py -n {} -a_nodata {} -co "COMPRESS=LZW" -of GTiff -o {} {}'.format(nodata,nodata,tempname,str_list))
    sat.gaivota_clip_raster(tempname,shp_st,outname)
    os.remove(tempname)
    
    if(remove_files==True):
        for f in flist:
            os.remove(f)
        
    return outname

###########################################################

# third part - Reconstruct Time-Series from Amplitude and Phase rasterssecond part - appling hants for each states of regions
def call_hants_recon_parallel(year,st,fea,gtile,flist,path_hants,feaToMinLimit,freq,
                        freq_recon,feaToErrorLimit,feaToOutlier ):
    initime=datetime.datetime.now()

    #==========================================================================
    #loading data
    
    path_feature_hants=os.path.join(path_hants,'amp_pha_'+str(freq)+'freq')
    path_feature_hants_res=os.path.join(path_hants,'reconstructed_'+str(freq_recon)+'freq')
    sd.check_dir(path_feature_hants)
    sd.check_dir(path_feature_hants_res)
    
    amp_list=sorted(glob.glob(os.path.join(path_feature_hants,'*am*.tif')))
    ph_list=sorted(glob.glob(os.path.join(path_feature_hants,'*ph*.tif')))

#     print ('stacking arrays...',st,year)
    amp,gt,proj=sat.stack_tifs_as_arrays(amp_list)
    phase,gt,proj=sat.stack_tifs_as_arrays(ph_list)
    

    
    #==========================================================================
    # recompose time series
    yrecon=np.zeros([amp.shape[0],amp.shape[1],len(flist)])

    # AMP and PHASE resized to freq_recon+1 (axis 2)
    amp=amp[:,:,0:freq_recon+1]
    phase=phase[:,:,0:freq_recon+1]
    
#     print ('amp.shape:',amp.shape,st,year)
#     print ('ph.shape:',phase.shape,st,year)

#     print('yrecon.shape=',yrecon.shape)
    for i in range(yrecon.shape[0]):
#         if i%100==0:
#             print('recompose time series: i = ',i,' of ',yrecon.shape[0],'rows', st, year,gtile)
        for j in range(yrecon.shape[1]):
            yrecon[i,j,:]=(np.tile(amp[i,j,:],(yrecon.shape[2], 1)).T*np.cos((np.tile(np.arange(yrecon.shape[2]),(freq_recon+1, 1))*np.arange(freq_recon+1).reshape((-1,1))*2.0*np.pi/yrecon.shape[2])-np.tile(phase[i,j,:]*np.pi/180.0,(yrecon.shape[2],1)).T)).sum(axis=0)

    amp=None
    phase=None
    #==========================================================================
    # saving recomposition of time series
    yrecon[yrecon>1]=1
    yrecon[yrecon<feaToMinLimit[fea]]=feaToMinLimit[fea]

    vrt_list=[]
    print ('reconstructing',yrecon.shape[2],'files', st,'year='+str(year))
    for z in range(yrecon.shape[2]):

        fname=os.path.basename(flist[z])
        split=fname.split('_')            
        date_str=split[0]+'_'+split[1]+'_'+split[2]
        #"{0:0=3d}".format(z+1)
        #gdal_array_2_tif(yrecon[:,:,z]*10000,gt,proj,os.path.join(path_feature_hants_res,'yrecon'+str(freq_recon)+'_'+band+'.tif'))
        outname=os.path.join(path_feature_hants_res,os.path.basename(flist[z].replace('.tif','_hantsrecon_'+str(freq_recon)+'freq.tif')))
        vrt_list.append(outname)
        sat.array_2_raster(outname,yrecon[:,:,z]*10000,gt,proj,nodata=-28672,type_data='int16')

    yrecon=None

#     print('vrt_list:',len(vrt_list),'files')
#     print ('start_date:',vrt_list[0])
#     print ('final_date:', vrt_list[-1])
    vrt_recon=os.path.join(path_hants,fea+'_'+st+'_'+year+'_hants_'+str(freq_recon)+'freq_'+gtile+'.vrt')
#     print ('vrt_file:', vrt_list[-1])
    vrt=create_virtual_raster(vrt_list,vrt_recon)

    endtime=datetime.datetime.now()
#     print (st,year+' - elapsed time:',endtime-initime)
    print ('=== Hants-recon finished! - state=',st,'year=',year,'gtile=',gtile, 'gtile elapsed time=',endtime-initime)
    
    ########################################
    
    
def call_gaivota_clip_raster_gtile(tif,shp,gtile,outname,nodata=0.):
    """
    Clip raster to .shp features
    """
    
    ds=gdal.Open(tif)
    gt=ds.GetGeoTransform()
    df=gpd.read_file(shp)
    bounds=df.bounds

    # Get shapefile bounds in array col,row units and adjust to temp clip
    minx= (float(bounds.minx)-gt[0])/gt[1]
    minx_col=int(minx)-1

    maxx=(float(bounds.maxx)-gt[0])/gt[1]
    maxx_col=int(maxx)+1

    maxy=(float(bounds.maxy)-gt[3])/gt[5]
    maxy_row=int(maxy)-1

    miny=(float(bounds.miny)-gt[3])/gt[5]
    miny_row=int(miny)+1

    new_minx=gt[0]+minx_col*gt[1]
    new_maxx=gt[0]+maxx_col*gt[1]
    new_miny=gt[3]+miny_row*gt[5]
    new_maxy=gt[3]+maxy_row*gt[5]

    # Create temp raster clipped by adjusted bounds
    temp_tif=outname.replace('.tif','_temp_'+gtile+'.tif')
    os.system('gdal_translate -projwin {} {} {} {} -co "COMPRESS=LZW" -of GTiff {} {}'.format(new_minx,new_maxy,new_maxx,new_miny,os.path.abspath(tif),temp_tif))

    # get raster resolution
    res=sat.get_raster_resolution(temp_tif)

    # Clip temp raster to shapefile polygon - DO NOT USE -crop_to_cutline (prevents pixel grid shift)
    os.system('gdalwarp -dstnodata {} -q -cutline {} -tr {} {} -co "COMPRESS=LZW" -of GTiff {} {}'.format(nodata,shp,res,res,os.path.abspath(temp_tif),outname))
    os.remove(temp_tif)
    return outname


In [ ]:
# ############################################################
# #  Create Total Gtiles (1 deg)

# tb=[-62,-50,-25,-7] # Brazil xmin, xmax, ymin, ymax
# stepsize=0.10 # degrees



# path_gtiles='/media/data/projeto_monitoramento_v2/gtiles_'+str(stepsize)+'/'
# print path_gtiles

# sd.check_dir(path_gtiles)

# strtb='_'
# for x in range(len(tb)):
#     if(x<2):
#         if(tb[x]<0):
#             strtb=strtb+str(tb[x]).replace('-','')+'W_'
#         else:
#             strtb=strtb+str(tb[x])+'E_'
#     else:
#         if(tb[x]<0):
#             strtb=strtb+str(tb[x]).replace('-','')+'S_'
#         else:
#             strtb=strtb+str(tb[x])+'N_'

# gtiles_path=os.path.join(path_gtiles,'gaivota_tiles'+strtb+'stepsize_'+str((stepsize)).replace('.','_')+'deg.geojson')
# print ('Gaivota_tiles:',gtiles_path)
# if(os.path.isfile(gtiles_path)==False):
#     gtiles=create_gaivota_tiles(tb,stepsize)
#     gtiles.to_file(gtiles_path, driver='GeoJSON')
#     print ('Gaivota_tiles created!')
# else:
#     print ('--- Gaivota_tiles already exists! ---\n')
#     gtiles=gpd.read_file(gtiles_path)
    
# # display(gtiles.head())

# ############################################################

In [ ]:
# RUN HANTS - GTILES







##################################################################
# path='/media/data/projeto_monitoramento_v2/'
# path='/home/ubuntu/bruno_testes/2019-07-04_MT_Cotton_download_NDVI_MODIS_hants/'
path='/media/data/projeto_monitoramento_v2/'
# 

feaToMinLimit={'ndwi':0.0,'ndvi':-0.3,'savi':0.0,'evi':0.0,'albedo':0.0,'ndmi':0.0}
feaToErrorLimit={'ndwi':0.1,'ndvi':0.1,'savi':0.1,'evi':0.1,'albedo':0.02,'ndmi':0.1}
feaToOutlier={'ndwi':'Hi','ndvi':'Lo','savi':'Lo','evi':'Lo','albedo':'Hi','ndmi':'Hi'}

feaToMinLimit={'ndwi':0.0,'ndvi':0.0,'savi':0.0,'evi':0.0,'albedo':0.0,'ndmi':0.0}
feaToErrorLimit={'ndwi':0.1,'ndvi':0.1,'savi':0.1,'evi':0.1,'albedo':0.02,'ndmi':0.1}
feaToOutlier={'ndwi':'Hi','ndvi':'Lo','savi':'Lo','evi':'Lo','albedo':'Hi','ndmi':'Hi'}
##################################################################



#### FREQUENCIES CONSIDERED
freq=7 # frequencies_considered_count
freq_recon=4 # frequencies_considered_count to reconstruct time series


recon=True # Reconstruct time series in call hants function

# merge gtiles for amplitudes and phases
merge_gtiles=True

# reconstruct time-series and merge filtered gtiles
recon_gtiles=True
# merge_recon_gtiles=True


list_features=[
#     'evi',
    'ndvi',
#     'B05'
]

states=[
    'BA',
#     'CE',
#     'SP','MG','ES','RJ',
#     'GO',
#     'DF',
#     'MT',
#     'MS',
#     'PR','SC','RS',
# 'MA','PI',
#     'RN','PB','AL','SE','PE',
# 'PA',
#     'RO','RR','AP',
#     'AM',
#     'AC','TO',
# 'PRY',  
# 'URY',
#     'ARG',
]

years=[
#     '2009',
#        '2017',
# '2018',
# '2019',
# '2020',
# '2021',
# '2022',
#     '2023',
    '2024',
]

years_dates={
            '2009':['2008-09-01','2009-08-31'],
            '2017':['2016-09-01','2017-08-31'],
            '2018':['2017-09-01','2018-08-31'],
            '2019':['2018-09-01','2019-08-31'],
            '2020':['2019-09-01','2020-08-31'],
            '2021':['2020-09-01','2021-08-31'],
            '2022':['2021-09-01','2022-08-31'],
            '2023':['2022-09-01','2023-08-31'],
            '2024':['2023-09-01','2024-08-31'],

            }


############################################################
#  Create Total Gtiles (1 deg)

# tb=[-80,-20,-40,10] # Brazil
tb=[-180,180,-90,90] # Brazil
stepsize=1.0 # degrees


# tb=[-62,-50,-25,-7] # Brazil xmin, xmax, ymin, ymax
# stepsize=0.10 # degrees


path_shp='/media/data/projeto_monitoramento_v2/shapefiles/'
print(os.path.isdir(path_shp),path_shp)

path_gtiles='/media/data/projeto_monitoramento_v2/gtiles_'+str(stepsize)+'/'
print (os.path.isdir(path_gtiles),path_gtiles)

sd.check_dir(path_gtiles)


print('freq',freq)
print('freq_recon',freq_recon)
print('feature',feature)

for yy in years:
    
    print('year=',years,years_dates[yy])
print('states=',states)

numProcess = 4#int((multiprocessing.cpu_count())*.25)
print('numProcess',numProcess)

# Run!

In [ ]:
total_initime=datetime.datetime.now()
print ('total_initime',datetime.datetime.now())

strtb='_'
for x in range(len(tb)):
    if(x<2):
        if(tb[x]<0):
            strtb=strtb+str(tb[x]).replace('-','')+'W_'
        else:
            strtb=strtb+str(tb[x])+'E_'
    else:
        if(tb[x]<0):
            strtb=strtb+str(tb[x]).replace('-','')+'S_'
        else:
            strtb=strtb+str(tb[x])+'N_'

gtiles_path=os.path.join(path_gtiles,'gaivota_tiles'+strtb+'stepsize_'+str((stepsize)).replace('.','_')+'deg.geojson')
print ('Gaivota_tiles:',gtiles_path)
if(os.path.isfile(gtiles_path)==False):
    gtiles=create_gaivota_tiles(tb,stepsize)
    gtiles.to_file(gtiles_path, driver='GeoJSON')
    print ('Gaivota_tiles created!')
else:
    print ('--- Gaivota_tiles already exists! ---\n')
    gtiles=gpd.read_file(gtiles_path)
    
# display(gtiles.head())


############################################################


print '\n========================================================='
print '========================= Clip data  ===================='
print '========================================================='

for fea in list_features:
    
    print('feature:',fea)
    path_fea=os.path.join(path,fea)
    sd.check_dir(path_fea)
    for st in states:
        print ('state:',st)
        print('***',os.path.join(path_shp,'*'+st+'.shp'))
        
        shp=sorted(glob.glob(os.path.join(path_shp,'*'+st+'.shp')))
        print('shp',shp)
        if(len(shp)>0):
            print 'state_shapefile=',shp[0]
            path_st=os.path.join(path,fea,st)
            
            # gtiles for this state
            gtile_list=[]
            sd.check_dir(os.path.join(path_gtiles,st))
            gtiles_st=os.path.join(path_gtiles,st,st+'_gtiles_all_'+str(int(stepsize))+'deg.geojson')
            
            print ('gtiles_st:',gtiles_st)
            gtiles_st_df=select_gtiles_by_shp(gtiles_path,gtiles,shp[0])
            if(os.path.isfile(gtiles_st)==False):
                gtiles_st_df.to_file(gtiles_st, driver='GeoJSON')
            for line in range(len(gtiles_st_df)):
                gtname=gtiles_st.replace('_gtiles_all','_'+gtiles_st_df.loc[line,'gtile_name'])

#                 print gtname
                gtile_list.append(gtname)
                if(os.path.isfile(gtname)==False):
                    gtiles_st_df.loc[[line]].to_file(gtname, driver='GeoJSON')
            print ('gtiles for '+st+' state:',len(gtile_list))
            print('...')
            
            for y in years:
                y=str(y)
                path_st_y=os.path.join(path,fea,st,y)
                path_st_y_hants=os.path.join(path,fea+'_hants',st,y)
                sd.check_dir(path_st_y_hants)
                print path_st_y

                date_start=years_dates[y][0]
                date_end  =years_dates[y][1]
                
                date_range=pd.date_range(start=date_start,end=date_end)
                date_range=[str(item.date()).replace('-','_') for item in date_range]
                
                print ('YEAR:',y)
#                 print ('*** date_start:',date_start)
#                 print ('*** date_end:  ',date_end)
                
                flist=[]
                for dd in date_range:
                    xy=dd.split('_')[0] # ano do date_range item
                    temp_path=os.path.join(path,fea,st)
                    f=sorted(glob.glob(os.path.join(temp_path,xy,dd+'*'+st+'.tif')))
                    if(len(f)>0):
                        for item in f:
                            flist.append(item)


                #create original .vrt (total state)
                vrt_name=os.path.join(path_st_y_hants,fea+'_'+st+'_'+y+'_original.vrt')
                print (vrt_name,len(flist),'files to vrt')
                vrt=create_virtual_raster(flist,vrt_name)
                
                # Clip state data into gtiles
                print ('====')
                if(len(flist)>0):
                    for item in flist:
                        print ('clip:',os.path.basename(item),'-',len(gtile_list),'gtiles')
                         # criar pasta dos gtiles com tifs clipados dentro
                        for gtc in gtile_list:
#                             path_gtiles_tif=os.path.join(path_st_y,os.path.basename(gtc).replace('.geojson',''))
                            path_gtiles_tif=os.path.join(os.path.split(item)[0],os.path.basename(gtc).replace('.geojson',''))
                            sd.check_dir(path_gtiles_tif)
                            tname=os.path.basename(gtc).replace('.geojson','.tif').replace(st+'_','')
                            gtile_clipped_name=os.path.basename(item).replace('.tif','_'+tname)
                            
                            gtile_clipped_path=os.path.join(path_gtiles_tif,gtile_clipped_name )
#                             print gtile_clipped_path
#                             print '...'

                            xgtile=os.path.basename(gtc).split('.')[0]
                            if(os.path.isfile(gtile_clipped_path)==False):
#                                 sat.gaivota_clip_raster(item,gtc,gtile_clipped_path)
                                multiprocessing.Process(target=call_gaivota_clip_raster_gtile, 
                                                args=(item,gtc,xgtile,gtile_clipped_path,0.)).start()
    
while (len(multiprocessing.active_children()) > 0):
    time.sleep(1)

print ('Clip - Finished!')
print ('')


####################################################################################


print '\n========================================================='
print '================ Run gtiled hants ======================='
print '========================================================='

for fea in list_features:
    
    print('feature:',fea)
    path_fea=os.path.join(path,fea)
    sd.check_dir(path_fea)
    for st in states:
        print ('state:',st)
        
        
        shp=sorted(glob.glob(os.path.join(path_shp,'*'+st+'.shp')))

        if(len(shp)>0):
            print ('state_shapefile=',shp[0])
            path_st=os.path.join(path,fea,st)
            
            # gtiles for this state
            gtile_list=[]
            sd.check_dir(os.path.join(path_gtiles,st))
            gtiles_st=os.path.join(path_gtiles,st,st+'_gtiles_all_'+str(int(stepsize))+'deg.geojson')
            
            print ('gtiles_st:',gtiles_st)
            gtiles_st_df=select_gtiles_by_shp(gtiles_path,gtiles,shp[0])
            if(os.path.isfile(gtiles_st)==False):
                gtiles_st_df.to_file(gtiles_st, driver='GeoJSON')
            for line in range(len(gtiles_st_df)):
                gtname=gtiles_st.replace('_gtiles_all','_'+gtiles_st_df.loc[line,'gtile_name'])

#                 print gtname
                gtile_list.append(gtname)
                if(os.path.isfile(gtname)==False):
                    gtiles_st_df.loc[[line]].to_file(gtname, driver='GeoJSON')
            print ('gtiles for '+st+' state:',len(gtile_list))
            print('...')
            
            for y in years:
                y=str(y)
                path_st_y=os.path.join(path,fea,st,y)
                path_st_y_hants=os.path.join(path,fea+'_hants',st,y)
                sd.check_dir(path_st_y_hants)
                print (path_st_y)
                

                print('')
                print('List gtiled tifs:')
                for fclip in sorted(next(os.walk(path_st_y))[1]):
                    print fclip
                    fclip_list=[]
                    

                    # List files by date_range
                    date_start=years_dates[y][0]
                    date_end  =years_dates[y][1]

                    date_range=pd.date_range(start=date_start,end=date_end)
                    date_range=[str(item.date()).replace('-','_') for item in date_range]

                    print ('YEAR:',y)
                    print ('*** date_start:',date_start)
                    print ('*** date_end:  ',date_end)

                    for dd in date_range:
                        xy=dd.split('_')[0] # ano do date_range item
                        temp_path=os.path.join(path,fea,st)
                        f=sorted(glob.glob(os.path.join(temp_path,xy,fclip,dd+'*'+st+'*.tif')))
                        if(len(f)>0):
                            for item in f:
                                fclip_list.append(item)



                    if(len(fclip_list)>0):
                        print len(fclip_list),' files'

                        print ('start_date:',os.path.basename(fclip_list[0]))
                        print ('final_date:', os.path.basename(fclip_list[-1]))
                        print ('---')

                        path_hants=os.path.join(path_st_y_hants,fclip)
                        sd.check_dir(path_hants)

                        multiprocessing.Process(target=call_hants_parallel, 
                                                args=(y,st,fea,fclip,fclip_list,path_hants,feaToMinLimit,freq, 
                                                      freq_recon,feaToErrorLimit,feaToOutlier,recon)).start()


while (len(multiprocessing.active_children()) > 0):
    time.sleep(1)

print 'GTILE HANTS - Finished!'

########################################################################################3

print '\n========================================================='
print '================= MERGE AMPs and PHs GTILES ============='
print '========================================================='

for fea in list_features:
    print('feature:',fea)
    path_fea=os.path.join(path,fea+'_hants')
    print 'path_fea:',path_fea
    for st in states:
        print ('state:',st)
        shp=sorted(glob.glob(os.path.join(path_shp,'*'+st+'.shp')))

        if(len(shp)>0):
            print 'state_shapefile=',shp[0]
            path_st=os.path.join(path_fea,st)
            print 'path_st:',path_st
            for y in years:
                
                dict_files={}
                for f in range(freq+1):
                    dict_files['am'+str(f)]=[]
                    dict_files['ph'+str(f)]=[]
                
                path_fea_st_y=os.path.join(path_st,y)
                print 'path_fea_st_y:',path_fea_st_y
                gtile_folders=sorted(next(os.walk(path_fea_st_y))[1])
                gtile_folders
                for fclip in sorted(next(os.walk(path_fea_st_y))[1]):
                    for f in range(freq+1):
                    
                        flist_merge=sorted(glob.glob(os.path.join(path_fea_st_y,fclip,'amp_pha_'+str(freq)+'freq','*'+str(f)+'*'+fea+'*.tif')))
                        if(len(flist_merge)>0):
                            dict_files['am'+str(f)].append(flist_merge[0])
                            dict_files['ph'+str(f)].append(flist_merge[1])
                
                for k in sorted(dict_files.keys()):
                    
                    path_fea_st_y_hants=os.path.join(path_fea_st_y,'amp_pha_'+str(freq)+'freq')
#                     print path_fea_st_y_hants
                    sd.check_dir(path_fea_st_y_hants)
                    outname='hants_'+k+'_'+fea+'_'+st+'_'+y+'.tif'
                    outpath1=os.path.join(path_fea_st_y_hants,outname)
                    print fea,st,y,k,len(dict_files[k]),'files','- outname:',outname
                    
                    if(merge_gtiles==True):
                        multiprocessing.Process(target=call_merge_gtiles, 
                                                 args=(dict_files[k],shp[0],
                                                       outpath1,0.,False)).start()
while (len(multiprocessing.active_children()) > 0):
    time.sleep(1)

print 'MERGE GTILES - Finished!'


   
print '\n========================================================='
print '================= MERGE Hants Recon data ================'
print '========================================================='

for fea in list_features:
    print('feature:',fea)
    path_fea=os.path.join(path,fea+'_hants')
    print 'path_fea:',path_fea
    for st in states:
        print ('state:',st)
        shp=sorted(glob.glob(os.path.join(path_shp,'*'+st+'.shp')))

        if(len(shp)>0):
            print 'state_shapefile=',shp[0]
            path_st=os.path.join(path_fea,st)
            print ('path_st:',path_st)
            for y in years:
                dict_files={}               
                path_fea_st_y=os.path.join(path_st,y)
                print ('path_fea_st_y:',path_fea_st_y)
                gtile_folders=sorted(next(os.walk(path_fea_st_y))[1])
                gtile_folders=[k for k in gtile_folders if st in k]
                print ('GTILES:',gtile_folders)
                
                dates=[]
                for fclip in gtile_folders:
                    temp_fcliped_recon=sorted(glob.glob(os.path.join(path_fea_st_y,fclip,'reconstructed_'+str(freq_recon)+'freq','*.tif')))
                    print ('fcliped_recon:',len(temp_fcliped_recon),'files')
                    for d in temp_fcliped_recon:
                        dates.append(os.path.basename(d)[0:10])
                dates=np.unique(dates)
                
                for d in dates:
                    dlist=[]
                    for gtilex in gtile_folders:
                        fcliped_recon=sorted(glob.glob(os.path.join(path_fea_st_y,gtilex
                                                                    ,'reconstructed_'+str(freq_recon)+'freq',
                                                                    d+'*.tif')))[0]
                        dlist.append(fcliped_recon)
                    dict_files[d]=dlist

                for k in sorted(dict_files.keys()):
#                     print k
#                     print dict_files[k]
                    path_fea_st_y_hants=os.path.join(path_fea_st_y,'reconstructed_'+str(freq_recon)+'freq')
                    sd.check_dir(path_fea_st_y_hants)
                    outname=os.path.basename(dict_files[k][0]).replace(gtile_folders[0]+'_','')
                    outpath2=os.path.join(path_fea_st_y_hants,outname)
                    
#                     print (fea,st,y,k,len(dict_files[k]),'files','- outname:')
#                     print (outpath2)
    
                    if(merge_gtiles==True):
                        multiprocessing.Process(target=call_merge_gtiles, 
                                                 args=(dict_files[k],shp[0],
                                                       outpath2,0.,False)).start()
                print('----')
                                     
while (len(multiprocessing.active_children()) > 0):
    time.sleep(1)
    
print 'MERGE HANTS RECON - Finished!'

##########################################################################################################


print '\n================= Create Hants .VRT ================'
for fea in list_features:
    print('feature:',fea)
    
    print ('path_fea:',path_fea)
    for st in states:
        print ('state:',st)
        for y in years:
            print (y)
            path_fea_st_y=os.path.join(path,fea+'_hants',st,y)
            path_hants_recon=os.path.join(path,fea+'_hants',st,y,'reconstructed_'+str(freq_recon)+'freq')
            rec_list=sorted(glob.glob(os.path.join(path_hants_recon,'*.tif')))
            print('start:',rec_list[0])
            print('final:',rec_list[-1])
            print (len(rec_list),'files to VRT')
            vrt_name_final=os.path.join(path,fea+'_hants',st,y,fea+'_'+st+'_'+y+'_'+'hants_'+str(freq_recon)+'freq.vrt')
            print (vrt_name_final)
            create_virtual_raster(rec_list,vrt_name_final)
            print ('---')
            
            # delete folders
            gtile_folders=sorted(next(os.walk(path_fea_st_y))[1])
            gtile_folders=[k for k in gtile_folders if st in k]
#             print ('GTILES:',gtile_folders)
            
            for xfolder in gtile_folders:
#                 print os.path.join(path,fea+'_hants',st,y,xfolder)
                shutil.rmtree(os.path.join(path,fea+'_hants',st,y,xfolder))
                shutil.rmtree(os.path.join(path,fea,st,y,xfolder))
            print '=================================================================================='


total_endtime=datetime.datetime.now()
print ('\n====== Total elapsed time:',str(total_endtime-total_initime))

In [ ]:
# ('\n====== Total elapsed time:', '1:45:42.295619') 27 estados + PRY (2021) m5.24x